In [1]:
# 07_adversarial.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 07. Adversarial Debiasing (in-processing) — RCFS extension
#
# In-processing fairness via adversarial debiasing, implemented in PyTorch (gradient
# reversal). A predictor learns the target while an adversary tries to recover the
# sensitive attribute from the predictor's representation; reversing the adversary's
# gradient pushes the representation to be sensitive-attribute-invariant.
#
# Because in-processing is gradient-based, these models are neural networks — a different
# model family from the XGBoost-based baseline/reweighing/reject_option in notebook 04.
# For a fair comparison we also train a neural-network BASELINE (no adversary) and compare
# RCFS WITHIN the neural family. Results are reported separately from the XGBoost-based
# techniques. An ADV_LAMBDA sweep tests how the outcome depends on adversarial strength.
#
# RCFS is computed exactly as in notebook 04 (same component definitions and gamma).

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.autograd import Function

from sklearn.metrics import roc_auc_score
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

SHAP_SAMPLE = 500          # samples for the KernelExplainer on the NN
GAMMA_DEFAULT = 0.5
EXPL_KSTAR = 10
W_FAIR, W_EXPL = 1.0, 1.0

# Training hyperparameters
EPOCHS = 60
LR = 1e-3
ADV_LAMBDA = 1.0           # default adversarial strength for the main comparison
HIDDEN = 64
LAMBDA_GRID = [0.0, 0.1, 0.5, 1.0, 2.0]   # for the sweep

DATASETS = [
    ("German Credit",        "german",               "sex", "male"),
    ("Taiwan CC Default",    "taiwan",                "sex", "male"),
    ("Credit Card Approval", "credit_card_approval",  "sex", "male"),
]

# %%
# Helpers (RCFS components identical to notebook 04)
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

def di_spd(y_pred, priv_mask, unpriv_mask):
    ppr_p = y_pred[priv_mask].mean() if priv_mask.sum() else np.nan
    ppr_u = y_pred[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    if not (np.isfinite(ppr_p) and np.isfinite(ppr_u)) or ppr_p == 0:
        return np.nan, np.nan
    return ppr_u / ppr_p, ppr_u - ppr_p

def c_fair_score(test_df, sens_col, privileged, y_pred):
    s = test_df[sens_col]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values
    di, spd = di_spd(y_pred, priv_mask, unpriv_mask)
    if not np.isfinite(di):
        return np.nan
    return 0.5 * min(1.0, di) + 0.5 * (1.0 - abs(spd))

def c_expl_from_shap(shap_matrix, k_star=EXPL_KSTAR):
    mean_abs = np.abs(shap_matrix).mean(axis=0)
    total = mean_abs.sum()
    if total == 0:
        return 0.0
    p = mean_abs / total
    p = p[p > 0]
    if len(p) <= 1:
        return 1.0
    entropy = -np.sum(p * np.log(p))
    n_eff = np.exp(entropy)
    return float(min(1.0, k_star / n_eff))

def prevalence_threshold(proba, y_train):
    return np.quantile(proba, 1.0 - float(y_train.mean()))

def compliance_harmonic(cs, ws):
    pairs = [(c, w) for c, w in zip(cs, ws) if np.isfinite(c) and c > 0]
    if not pairs:
        return np.nan
    return sum(w for _, w in pairs) / sum(w / c for c, w in pairs)

def rcfs(c_perf, compliance, gamma):
    if not np.isfinite(compliance):
        return np.nan
    return (1 - gamma) * c_perf + gamma * compliance

# %%
# Gradient reversal layer: forward is identity, backward flips (and scales) the gradient
class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

def grad_reverse(x, lambda_):
    return GradReverse.apply(x, lambda_)

# Shared encoder -> predictor head (target) and adversary head (sensitive attribute)
class AdvDebiasNet(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.predictor = nn.Linear(hidden, 1)      # target logit
        self.adversary = nn.Linear(hidden, 1)      # sensitive-attribute logit

    def forward(self, x, adv_lambda=0.0):
        z = self.encoder(x)
        y_logit = self.predictor(z)
        s_logit = self.adversary(grad_reverse(z, adv_lambda))
        return y_logit.squeeze(1), s_logit.squeeze(1)

# %%
# Train a network. adv_lambda=0 -> plain NN baseline; adv_lambda>0 -> adversarial debiasing.
def train_net(X_train, y_train, s_train_bin, in_dim, adv_lambda, class_pos_weight,
              epochs=EPOCHS, lr=LR, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    net = AdvDebiasNet(in_dim)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    bce_y = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([class_pos_weight], dtype=torch.float32))
    bce_s = nn.BCEWithLogitsLoss()

    Xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    st = torch.tensor(s_train_bin, dtype=torch.float32)

    net.train()
    for _ in range(epochs):
        opt.zero_grad()
        y_logit, s_logit = net(Xt, adv_lambda)
        loss_y = bce_y(y_logit, yt)
        loss_s = bce_s(s_logit, st) if adv_lambda > 0 else torch.tensor(0.0)
        loss = loss_y + (loss_s if adv_lambda > 0 else 0.0)
        loss.backward()
        opt.step()
    net.eval()
    return net

def predict_proba_net(net, X):
    with torch.no_grad():
        logit, _ = net(torch.tensor(X, dtype=torch.float32), adv_lambda=0.0)
        return torch.sigmoid(logit).numpy()

# %%
# SHAP for the NN predictor via KernelExplainer on a background sample.
def shap_matrix_net(net, X_bg, X_eval, nsample=SHAP_SAMPLE, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    bg = X_bg[rng.choice(len(X_bg), size=min(50, len(X_bg)), replace=False)]
    ev = X_eval[rng.choice(len(X_eval), size=min(nsample, len(X_eval)), replace=False)]
    def f(data):
        with torch.no_grad():
            logit, _ = net(torch.tensor(data, dtype=torch.float32), adv_lambda=0.0)
            return torch.sigmoid(logit).numpy()
    explainer = shap.KernelExplainer(f, bg)
    sv = explainer.shap_values(ev, nsamples=100, silent=True)
    return np.array(sv)

# %%
# Compute all RCFS components for a trained net on one dataset
def score_net(net, test, X_train, X_test, y_test, y_train, sens_col, privileged):
    proba = predict_proba_net(net, X_test)
    thr = prevalence_threshold(proba, y_train)
    y_pred = (proba >= thr).astype(int)

    auc = roc_auc_score(y_test.values, proba)
    c_perf = max(0.0, (auc - 0.5) / 0.5)
    c_fair = c_fair_score(test, sens_col, privileged, y_pred)
    sv = shap_matrix_net(net, X_train, X_test)
    c_expl = c_expl_from_shap(sv)

    compliance = compliance_harmonic([c_fair, c_expl], [W_FAIR, W_EXPL])
    score = rcfs(c_perf, compliance, GAMMA_DEFAULT)
    return dict(AUC=auc, c_perf=c_perf, c_fair=c_fair, c_expl=c_expl,
                Compliance=compliance, RCFS=score)

# %%
# Main comparison: neural baseline vs adversarial debiasing (at ADV_LAMBDA)
rows = []
for display_name, key, sens_col, privileged in DATASETS:
    train, test = load_split(key)
    X_train_df, y_train = get_Xy(train, key)
    X_test_df, y_test = get_Xy(test, key)
    X_test_df = X_test_df[X_train_df.columns]

    X_train = X_train_df.values.astype(np.float32)
    X_test = X_test_df.values.astype(np.float32)
    in_dim = X_train.shape[1]
    s_train = (train[sens_col].values == privileged).astype(np.float32)

    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    pos_w = (n_neg / n_pos) if n_pos > 0 else 1.0

    for tech, lam in [("nn_baseline", 0.0), ("adversarial", ADV_LAMBDA)]:
        net = train_net(X_train, y_train.values.astype(np.float32),
                        s_train, in_dim, lam, pos_w)
        m = score_net(net, test, X_train, X_test, y_test, y_train, sens_col, privileged)
        m.update({"dataset": display_name, "technique": tech, "model_family": "neural_net"})
        rows.append(m)
        print(f"[{display_name:>22}] {tech:<12} AUC={m['AUC']:.3f}  "
              f"c_fair={m['c_fair']:.3f}  c_expl={m['c_expl']:.3f}  "
              f"Compliance={m['Compliance']:.3f}  RCFS={m['RCFS']:.3f}")

adv_df = pd.DataFrame(rows)[
    ["dataset", "technique", "model_family", "AUC", "c_perf", "c_fair",
     "c_expl", "Compliance", "RCFS"]
].round(4)
print(adv_df.to_string(index=False))
adv_df.to_csv(os.path.join(TAB_DIR, "07_adversarial_rcfs.csv"), index=False)

# %%
# Within-family comparison: does adversarial debiasing beat the neural baseline?
comp = (adv_df.pivot(index="dataset", columns="technique", values="RCFS")
        .reindex(index=["German Credit", "Taiwan CC Default", "Credit Card Approval"]))
comp["adversarial_beats_baseline"] = comp["adversarial"] > comp["nn_baseline"]
print(comp.to_string())
comp.to_csv(os.path.join(TAB_DIR, "07_adversarial_comparison.csv"))

# %%
# --- ADV_LAMBDA sweep ---
# Vary the adversarial strength to test whether the per-dataset outcome is driven by
# signal strength rather than a single lambda choice. lambda=0 reproduces the baseline.
sweep_rows = []
for display_name, key, sens_col, privileged in DATASETS:
    train, test = load_split(key)
    X_train_df, y_train = get_Xy(train, key)
    X_test_df, y_test = get_Xy(test, key)
    X_test_df = X_test_df[X_train_df.columns]

    X_train = X_train_df.values.astype(np.float32)
    X_test = X_test_df.values.astype(np.float32)
    in_dim = X_train.shape[1]
    s_train = (train[sens_col].values == privileged).astype(np.float32)

    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    pos_w = (n_neg / n_pos) if n_pos > 0 else 1.0

    for lam in LAMBDA_GRID:
        net = train_net(X_train, y_train.values.astype(np.float32),
                        s_train, in_dim, lam, pos_w)
        m = score_net(net, test, X_train, X_test, y_test, y_train, sens_col, privileged)
        m.update({"dataset": display_name, "adv_lambda": lam})
        sweep_rows.append(m)
        print(f"[{display_name:>22}] lambda={lam:<4} AUC={m['AUC']:.3f}  "
              f"c_fair={m['c_fair']:.3f}  RCFS={m['RCFS']:.3f}")

adv_sweep_df = pd.DataFrame(sweep_rows)[
    ["dataset", "adv_lambda", "AUC", "c_perf", "c_fair", "c_expl", "Compliance", "RCFS"]
].round(4)
print(adv_sweep_df.to_string(index=False))
adv_sweep_df.to_csv(os.path.join(TAB_DIR, "07_adversarial_lambda_sweep.csv"), index=False)

# %%
# Sweep summary: for each dataset, does any lambda>0 beat the lambda=0 baseline?
summary = []
for ds in ["German Credit", "Taiwan CC Default", "Credit Card Approval"]:
    sub = adv_sweep_df[adv_sweep_df["dataset"] == ds].set_index("adv_lambda")
    base_rcfs = sub.loc[0.0, "RCFS"]
    pos = sub.drop(index=0.0)
    best_lam = pos["RCFS"].idxmax()
    summary.append({
        "dataset": ds,
        "baseline_RCFS(l=0)": round(base_rcfs, 4),
        "best_lambda>0": best_lam,
        "best_RCFS": round(pos.loc[best_lam, "RCFS"], 4),
        "any_lambda_beats_baseline": bool(pos["RCFS"].max() > base_rcfs),
        "c_fair_at_best": round(pos.loc[best_lam, "c_fair"], 4),
        "AUC_at_best": round(pos.loc[best_lam, "AUC"], 4),
    })
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(TAB_DIR, "07_adversarial_lambda_summary.csv"), index=False)

[         German Credit] nn_baseline  AUC=0.816  c_fair=0.966  c_expl=0.316  Compliance=0.477  RCFS=0.554
[         German Credit] adversarial  AUC=0.806  c_fair=0.985  c_expl=0.360  Compliance=0.528  RCFS=0.570
[     Taiwan CC Default] nn_baseline  AUC=0.746  c_fair=0.948  c_expl=0.864  Compliance=0.904  RCFS=0.698
[     Taiwan CC Default] adversarial  AUC=0.732  c_fair=0.952  c_expl=0.903  Compliance=0.927  RCFS=0.695
[  Credit Card Approval] nn_baseline  AUC=0.573  c_fair=0.896  c_expl=0.390  Compliance=0.544  RCFS=0.344
[  Credit Card Approval] adversarial  AUC=0.529  c_fair=0.600  c_expl=0.650  Compliance=0.624  RCFS=0.341
             dataset   technique model_family    AUC  c_perf  c_fair  c_expl  Compliance   RCFS
       German Credit nn_baseline   neural_net 0.8162  0.6323  0.9665  0.3163      0.4766 0.5544
       German Credit adversarial   neural_net 0.8059  0.6117  0.9851  0.3605      0.5278 0.5698
   Taiwan CC Default nn_baseline   neural_net 0.7456  0.4912  0.9482  0.8641